# 06: Balanced class weights + combo features under the correct metric

**The correction driving this notebook:** the competition metric is **balanced accuracy**
(per the Evaluation page: "Submissions are evaluated on balanced accuracy between the
predicted class and the observed target"), not plain accuracy. All prior notebooks
scored with `accuracy`, which explains the CV/LB gap: notebook 05's model scored
0.96658 accuracy locally but 0.87097 balanced accuracy on the leaderboard. Re-scoring
the same model locally with `balanced_accuracy` gives 0.87165, matching the LB to
0.0007. The validation was never broken; it was measuring the wrong thing.

**Changes in this notebook:**
1. All scoring uses `balanced_accuracy`
2. `class_weight='balanced'` on the classifier, so minority recall counts
3. Three engineered features adapted from [Rugved Bane's public notebook](https://www.kaggle.com/code/rugvedbane/student-health-risk-lightgbm-optuna-0-95014)
   (0.95014 LB): `stress_activity_combo`, `activity_score`, `sleep_activity_ratio`
4. Dropped `diet_type` and `gender` (no class signal in crosstabs, notebook eda;
   independently dropped by Rugved)

**Key detail on the combo feature:** it is built with `.astype(str)` *before* encoding,
which converts NaN to the literal string "nan" — so combinations like "nan_active"
become real category levels. The missingness of the two strongest features is
preserved as signal even though the raw columns' NaN are handled by the encoder.

**Success criterion:** not the CV number itself, but CV/LB agreement at a higher level.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')

X = train.drop(columns=['id', 'health_condition'])
y = train['health_condition']
X_test = test.drop(columns=['id'])

## Feature engineering

Applied identically to train and test. The combo feature must be created before
any encoding so NaN values become "nan" string categories.

In [2]:
def add_features(df):
    df = df.copy()
    df['stress_activity_combo'] = (df['stress_level'].astype(str) + '_' +
                                   df['physical_activity_level'].astype(str))
    df['activity_score'] = df['step_count'] * df['exercise_duration']
    df['sleep_activity_ratio'] = df['sleep_duration'] / (df['exercise_duration'] + 1)
    return df.drop(columns=['diet_type', 'gender'])

X = add_features(X)
X_test = add_features(X_test)

print(X['stress_activity_combo'].value_counts().head(12))

stress_activity_combo
medium_sedentary    85035
medium_moderate     83116
medium_active       79816
high_moderate       58844
high_sedentary      56764
low_active          54631
low_moderate        52816
high_active         52720
low_sedentary       51419
nan_sedentary       26566
nan_moderate        26265
nan_active          25475
Name: count, dtype: int64


## Pipeline

Same encode-preserving-NaN structure as notebook 05, with two changes:
`class_weight='balanced'` on the classifier and `balanced_accuracy` scoring.
The combo feature joins the categorical list.

In [3]:
cat_cols = ['stress_level', 'sleep_quality', 'physical_activity_level',
            'smoking_alcohol', 'stress_activity_combo']
num_cols = [c for c in X.columns if c not in cat_cols]

encode = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan,
                           encoded_missing_value=np.nan), cat_cols),
    ('num', 'passthrough', num_cols),
])

model = Pipeline([
    ('encode', encode),
    ('clf', HistGradientBoostingClassifier(class_weight='balanced', random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
print(scores.round(5))
print(f"mean: {scores.mean():.5f}  notebook 05 rescored: 0.87165, its LB: 0.87097")

[0.94974 0.95112 0.94856 0.94881 0.94803]
mean: 0.94925  notebook 05 rescored: 0.87165, its LB: 0.87097


## Fit on full training data and generate submission

In [4]:
model.fit(X, y)
preds = model.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'health_condition': preds})
submission.to_csv('../data/submission.csv', index=False)
print(submission['health_condition'].value_counts(normalize=True))

health_condition
at-risk      0.810558
unhealthy    0.115495
fit          0.073947
Name: proportion, dtype: float64
